# LangChain Streaming

<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>
<b>About</b><br><br>

This notebook is derived from the following notebook, with modifications and extensions: https://github.com/AI-Engineering-bootcamp/ai-eng-nbs-public/blob/master/langchain-streaming-202503.ipynb
</div>

## Intro


Streaming means returning an LLM's output incrementally — token by token (or chunk by chunk) — as it's generated, instead of waiting for the entire response to finish before showing anything to the user.

**Why it matters**:
- **Perceived latency**: LLMs can take several seconds (or longer) to generate a full response. Streaming lets users see the first words almost immediately, making the application feel far more responsive even though the total generation time is the same.
- **Better UX for long outputs**: For long-form answers, reports, or code generation, users can start reading and reacting before the response is complete, rather than staring at a blank screen or loading spinner.
- **Early cancellation**: If a response is clearly going off-track, a user (or system) can stop generation early, saving time and cost.
- **Real-time interactivity**: Streaming is essential for chat-like interfaces (e.g., ChatGPT-style UIs), voice assistants, and any application where responsiveness is part of the product experience.

**Where it gets harder:**

Streaming a plain LLM call is straightforward — you just forward tokens as they arrive. But it becomes more complex once you introduce Agents, which run their own internal logic (tool calls, reasoning steps, intermediate "thoughts") before producing a final answer. Naively streaming an agent's raw output exposes all of that internal machinery (JSON actions, tool calls, etc.) instead of just the final, user-facing answer — so extra work is needed to filter and expose only what's relevant.

We'll start simple, streaming LLM output straight to the terminal, then work up to the more advanced case of streaming an Agent's response through a FastAPI backend.


<br>

## Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [1]:
# !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"

In [2]:
# !pip install "numexpr==2.14.2"

<br>

## Initial Setup



In [3]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


<br>

## LLM Streaming to Stdout

The simplest form of streaming is to print each token as it's generated. To do this, we initialize an LLM (one that supports streaming — not all do) with two parameters:

* `streaming=True` — enables streaming.
* `callbacks=[SomeCallbackHere()]` — a LangChain callback (or list of callbacks) that runs code every time a new token arrives.

Here we use LangChain's built-in `StreamingStdOutCallbackHandler`, which simply prints each token as it's generated.



In [4]:
import os
from langchain_openai import ChatOpenAI
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler


llm = ChatOpenAI(
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0.0,
    model_name="gpt-3.5-turbo",
    streaming=True,  # ! important
    callbacks=[StreamingStdOutCallbackHandler()]  # ! important
)

**How it works**

1. `streaming=True` tells OpenAI: **"Don't wait until you've finished the entire answer—send it to me one small piece (token) at a time."**
2. Each time a new token arrives, LangChain immediately passes it to the callback we provided (in this case, `StreamingStdOutCallbackHandler`).

3. `StreamingStdOutCallbackHandler` simply prints each token to the terminal as soon as it receives it.


<br>

Now if we call the LLM, we'll see the response being _streamed_ token by token in the output below, instead of appearing all at once.

In [5]:
llm.invoke("Tell a long story where Willy Wonka invites the world's best AI engineers")

Willy Wonka, the eccentric and enigmatic owner of the world-famous Wonka Chocolate Factory, had always been known for his innovative and out-of-the-box thinking. So when he decided to host a competition to find the world's best AI engineers, the tech world was abuzz with excitement.

AI engineers from all corners of the globe flocked to the factory, eager to showcase their skills and compete for the coveted title of "Wonka's AI Genius." The competition was fierce, with engineers working tirelessly to create the most advanced and groundbreaking AI technology.

Willy Wonka himself was impressed by the talent and creativity on display. He marveled at the engineers' ability to push the boundaries of what was possible with artificial intelligence, creating machines that could think, learn, and even feel emotions.

As the competition drew to a close, Wonka invited the finalists to a grand banquet in the factory's famous chocolate room. The room was filled with all manner of delectable treats

AIMessage(content='Willy Wonka, the eccentric and enigmatic owner of the world-famous Wonka Chocolate Factory, had always been known for his innovative and out-of-the-box thinking. So when he decided to host a competition to find the world\'s best AI engineers, the tech world was abuzz with excitement.\n\nAI engineers from all corners of the globe flocked to the factory, eager to showcase their skills and compete for the coveted title of "Wonka\'s AI Genius." The competition was fierce, with engineers working tirelessly to create the most advanced and groundbreaking AI technology.\n\nWilly Wonka himself was impressed by the talent and creativity on display. He marveled at the engineers\' ability to push the boundaries of what was possible with artificial intelligence, creating machines that could think, learn, and even feel emotions.\n\nAs the competition drew to a close, Wonka invited the finalists to a grand banquet in the factory\'s famous chocolate room. The room was filled with al

<br>

That was surprisingly easy — but streaming gets more complicated once we introduce agents, which reason over multiple steps and call tools before producing a final answer. Let's initialize an agent to see why.

In [ ]:
from langchain.memory import ConversationBufferWindowMemory
from langchain.agents import load_tools, AgentType, initialize_agent

# initialize conversational memory
memory = ConversationBufferWindowMemory(
    memory_key="chat_history",
    k=5,
    return_messages=True,
    output_key="output"
)

# create a single tool to see how it impacts streaming
# (we'll use LangChain's built-in LLM math tool)
tools = load_tools(["llm-math"], llm=llm)

# initialize the agent
agent = initialize_agent(
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    tools=tools,
    llm=llm,
    memory=memory,
    verbose=False, # set to True for detailed logging of agent steps
    max_iterations=3,
    early_stopping_method="generate",
    return_intermediate_steps=False
)

<br>

The agent reuses the same `llm` object, so it already has our `StreamingStdOutCallbackHandler` attached. Let's see what streaming looks like when running the agent.

In [7]:
prompt = "Hello, how are you?"

result_1 = agent(prompt)

# print(result_1) # we can keep this commented ('StreamingStdOutCallbackHandler' will automatically print each token in the terminal)

/var/folders/x1/00w2xr_j197gk698c1ljm3300000gn/T/ipykernel_75054/150748627.py:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  result_1 = agent(prompt)


```json
{
    "action": "Final Answer",
    "action_input": "I'm just a computer program, so I don't have feelings, but I'm here and ready to assist you. How can I help you today?"
}
```

<br>

Not bad, but now we're streaming the LLM's _entire_ raw output — including the JSON structure the agent uses internally to decide on tool calls and reasoning steps. This is useful for debugging, but not something we'd want to show end users. 

Let's see another example where we ask a math question that requires a tool call:

In [8]:
result_2 = agent("what is the square root of 71?")

# print(result_2)

```json
{
    "action": "Calculator",
    "action_input": "square root of 71"
}
``````text
71**0.5
```
...numexpr.evaluate("71**0.5")...
```json
{
    "action": "Final Answer",
    "action_input": "The square root of 71 is approximately 8.426149773176359."
}
```

Looking at the streamed output above, we can see the raw JSON "thoughts" the agent streams token-by-token as it works through the problem.

This is exactly the problem we mentioned earlier: streaming shows us all of this internal JSON reasoning, when really we'd only want to stream the final answer text to an end user.

<br>

The raw output can be very useful in some cases (e.g. debugging), but in many other cases you only want to stream the final answer. 

We have two options: 
- Write a custom callback handler
- Use LangChain's purpose-built `FinalStreamingStdOutCallbackHandler`. 

<br>

Let's try the built-in one first. Before changing anything, let's just inspect the LLM's current `callbacks` (still our `StreamingStdOutCallbackHandler` from before):

In [9]:
from langchain.callbacks.streaming_stdout_final_only import FinalStreamingStdOutCallbackHandler

# Configure the LLM to begin streaming only after the model emits the
# tokens ["Final", "Answer"].
agent.agent.llm_chain.llm.callbacks = [
    FinalStreamingStdOutCallbackHandler(
        answer_prefix_tokens=["Final", "Answer"]
    )
]

Let's try it out:

In [10]:
result_3 = agent("what is the square root of 71?")

# print(result_3)

",
    "action_input": "The square root of 71 is approximately 8.426149773176359."
}
```

<br>

Still not quite clean — tuning `answer_prefix_tokens` correctly is finicky. It's usually easier to write a custom callback that filters tokens ourselves, only printing once it detects we're inside the `action_input` of the final answer:

In [12]:
import sys
from types import MethodType

# `state` holds the running content and a flag marking whether we've reached the final answer.
state = {"content": "", "final_answer": False}

def on_llm_new_token(self, token, **kwargs):
    state["content"] += token
    if "Final Answer" in state["content"]:
        # now we're in the final answer section, but don't print yet
        state["final_answer"] = True
        state["content"] = ""
    if state["final_answer"]:
        if '"action_input": "' in state["content"]:
            # strip stray quote/brace characters left over from the surrounding JSON structure
            clean_token = token.replace('"', "").replace("}", "")
            if clean_token:
                sys.stdout.write(clean_token)  # equal to `print(clean_token, end="")`
                sys.stdout.flush()

callback_handler = StreamingStdOutCallbackHandler()
# Bind our function as the handler's `on_llm_new_token` method, so LangChain calls it
# the same way it would call any callback method: `callback_handler.on_llm_new_token(token, ...)`.
callback_handler.on_llm_new_token = MethodType(on_llm_new_token, callback_handler)

agent.agent.llm_chain.llm.callbacks = [callback_handler]

<br>

Let's try again with our custom handler:

In [13]:
result_4 = agent("what is the square root of 71?")

# print(result_4)

 The square root of 71 is approximately 8.426149773176359.

```